[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/17_Frequency_Response.ipynb)

# DiveLab

## Notebook 17 — Frequency Response: Gain, Phase, Breathing and Delay

**Guiding question:** How does the diver-control system respond to disturbances that occur at different speeds?

Frequency response asks what happens when the input oscillates at frequency \(\omega\).

This notebook also introduces an important distinction:

> **Breathing can act both as a periodic disturbance and as a limited fast control input.**

We introduce both roles here, then develop breathing dynamics fully in Notebook 18.

## Learning objectives

By the end of this notebook, you will be able to:

- explain frequency response in simple words;
- evaluate \(G(j\omega)\);
- interpret gain and phase;
- read Bode magnitude and phase plots;
- understand bandwidth;
- interpret delay as phase lag;
- understand gain and phase margin conceptually;
- connect PID with frequency response;
- model breathing as both disturbance and control input;
- understand why breathing and BCD naturally operate on different time scales.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Why frequency matters

A control system may respond well to slow changes but poorly to fast ones.

So stability alone is not enough.

We also ask:

> How strongly does the system respond to disturbances at different frequencies?

# 2. Sinusoidal input

Consider:

\[
u(t)=A\sin(\omega t)
\]

where \(\omega\) is angular frequency in rad/s.

Frequency in hertz is:

\[
f=\frac{\omega}{2\pi}.
\]

In [ ]:
omega = 2.0
t = np.linspace(0, 10, 1000)
u = np.sin(omega*t)

plt.plot(t, u)
plt.xlabel("Time [s]")
plt.ylabel("Input")
plt.title("Sinusoidal input")
plt.grid(True)
plt.show()

# 3. Sinusoidal steady-state response

For a stable linear time-invariant system, a sinusoidal input eventually produces a sinusoidal output at the same frequency:

\[
u(t)=A\sin(\omega t)
\]

\[
y(t)=A|G(j\omega)|\sin(\omega t+\phi(\omega)).
\]

The system changes:

- amplitude by \(|G(j\omega)|\);
- phase by \(\phi(\omega)\).

# 4. Frequency response from a transfer function

Given:

\[
G(s),
\]

set:

\[
s=j\omega.
\]

Then:

\[
\boxed{G(j\omega)}
\]

is the frequency response.

# 5. First-order example

Let:

\[
G(s)=\frac{1}{\tau s+1}.
\]

Then:

\[
G(j\omega)=\frac{1}{1+j\omega\tau}.
\]

Magnitude:

\[
|G(j\omega)|
=
\frac{1}{\sqrt{1+(\omega\tau)^2}}.
\]

Phase:

\[
\phi(\omega)
=
-\tan^{-1}(\omega\tau).
\]

In [ ]:
tau = 1.0
omega_grid = np.logspace(-2, 2, 500)

G = 1/(1 + 1j*omega_grid*tau)

magnitude = np.abs(G)
phase_deg = np.angle(G, deg=True)

In [ ]:
plt.semilogx(omega_grid, magnitude)
plt.xlabel("Angular frequency ω [rad/s]")
plt.ylabel("|G(jω)|")
plt.title("Magnitude response")
plt.grid(True)
plt.show()

At low frequency, the system follows the input well.

At high frequency, the output is attenuated.

This is **low-pass behavior**.

In [ ]:
plt.semilogx(omega_grid, phase_deg)
plt.xlabel("Angular frequency ω [rad/s]")
plt.ylabel("Phase [degrees]")
plt.title("Phase response")
plt.grid(True)
plt.show()

At low frequency the phase is near \(0^\circ\).

As frequency increases, the output increasingly lags behind the input.

# 6. Decibels and Bode magnitude

Bode plots usually express magnitude in decibels:

\[
M_{\mathrm{dB}}
=
20\log_{10}|G(j\omega)|.
\]

In [ ]:
mag_db = 20*np.log10(magnitude)

plt.semilogx(omega_grid, mag_db)
plt.xlabel("Angular frequency ω [rad/s]")
plt.ylabel("Magnitude [dB]")
plt.title("Bode magnitude")
plt.grid(True)
plt.show()

# 7. Bode plots

A Bode representation consists of:

1. magnitude versus logarithmic frequency;
2. phase versus logarithmic frequency.

Together they show how the system reshapes oscillatory inputs across frequency.

# 8. Time-domain check at three frequencies

In [ ]:
def simulate_first_order_sine(omega, tau=1.0, duration=40.0, dt=0.002):
    t = np.arange(0, duration+dt, dt)
    u = np.sin(omega*t)
    y = np.zeros_like(t)

    for k in range(len(t)-1):
        ydot = (u[k] - y[k])/tau
        y[k+1] = y[k] + ydot*dt

    return t, u, y

for w in [0.2, 1.0, 5.0]:
    tt, uu, yy = simulate_first_order_sine(w)

    plt.figure()
    plt.plot(tt, uu, label="Input")
    plt.plot(tt, yy, label="Output")
    plt.xlim(20, 40)
    plt.xlabel("Time [s]")
    plt.ylabel("Amplitude")
    plt.title(f"Response at ω = {w} rad/s")
    plt.grid(True)
    plt.legend()
    plt.show()

Slow oscillations are followed closely.

Faster oscillations show:

- smaller output amplitude;
- larger phase lag.

The time-domain and frequency-domain views are describing the same system.

# 9. Bandwidth

For a first-order low-pass system, the \(-3\) dB frequency is:

\[
\omega_B=\frac{1}{\tau}.
\]

At this point:

\[
|G|=\frac{1}{\sqrt2}.
\]

This gives a useful measure of how rapidly a system can respond.

In [ ]:
bandwidth = 1/tau
print(f"Bandwidth = {bandwidth:.3f} rad/s")
print(f"Bandwidth = {bandwidth/(2*np.pi):.3f} Hz")

# 10. Control bandwidth

A feedback loop also has an effective bandwidth.

Very roughly:

- below bandwidth: tracking and disturbance rejection can be effective;
- well above bandwidth: the controller cannot react strongly enough or quickly enough.

A human diver also has finite effective control bandwidth because perception, decision and action are not instantaneous.

# 11. Delay as phase lag

A pure delay is:

\[
e^{-sT}.
\]

At:

\[
s=j\omega,
\]

we obtain:

\[
e^{-j\omega T}.
\]

Its magnitude is 1, but phase is:

\[
\boxed{\phi_{\text{delay}}=-\omega T}
\]

in radians.

In [ ]:
T_delay = 0.8

phase_delay_deg = np.rad2deg(-omega_grid*T_delay)

plt.semilogx(omega_grid, phase_delay_deg)
plt.xlabel("Angular frequency ω [rad/s]")
plt.ylabel("Phase [degrees]")
plt.title("Pure delay adds increasing phase lag")
plt.grid(True)
plt.show()

This explains why the same reaction delay becomes more problematic for faster control activity.

# 12. Negative feedback and phase

Negative feedback works because corrective action opposes error.

But enough phase lag can make a correction arrive so late that it reinforces rather than opposes the current motion.

Near \(180^\circ\) of total phase shift, negative feedback can behave like positive feedback.

# 13. Loop transfer function

For plant \(G(s)\) and controller \(C(s)\):

\[
L(s)=C(s)G(s).
\]

Classical stability analysis studies:

\[
L(j\omega).
\]

# 14. Gain crossover and phase margin

Gain crossover occurs when:

\[
|L(j\omega_{gc})|=1.
\]

Phase margin is:

\[
PM=180^\circ+\angle L(j\omega_{gc}).
\]

A positive phase margin indicates some tolerance to additional phase lag.

# 15. Gain margin

At the frequency where loop phase reaches \(-180^\circ\), gain margin asks how much the loop gain could increase before reaching unity.

Gain and phase margins are measures of **robustness**, not just nominal stability.

# 16. Breathing enters frequency response

Breathing changes lung volume.

Lung volume changes displaced water volume.

Therefore breathing changes buoyancy.

It can appear in two distinct ways:

\[
\boxed{\text{breathing as disturbance}}
\]

and

\[
\boxed{\text{breathing as control}}.
\]

## Breathing as disturbance

Ordinary cyclic breathing can be approximated as:

\[
\Delta V_L(t)
=
A_L\sin(\omega_b t).
\]

Then:

\[
\Delta F_B(t)
=
\rho g\,\Delta V_L(t).
\]

From the depth-control viewpoint, this is a periodic buoyancy disturbance.

In [ ]:
rho_water = 1025.0
g = 9.80665

A_lung = 0.0004   # 0.4 L illustrative amplitude
f_breath = 0.20   # Hz, illustrative
omega_breath = 2*np.pi*f_breath

tb = np.linspace(0, 30, 1500)

delta_V = A_lung*np.sin(omega_breath*tb)
delta_F = rho_water*g*delta_V

plt.plot(tb, delta_F)
plt.xlabel("Time [s]")
plt.ylabel("Buoyancy variation [N]")
plt.title("Simplified cyclic breathing disturbance")
plt.grid(True)
plt.show()

The numerical values are illustrative. The goal here is to visualize the systems concept.

Notebook 18 will model breathing dynamics more explicitly.

# 17. Breathing frequency on the Bode plot

In [ ]:
plt.semilogx(omega_grid, mag_db)
plt.axvline(omega_breath, linestyle="--", label="Illustrative breathing frequency")

plt.xlabel("Angular frequency ω [rad/s]")
plt.ylabel("Magnitude [dB]")
plt.title("Breathing frequency relative to plant response")
plt.grid(True)
plt.legend()
plt.show()

This suggests an important question:

> Is breathing inside the effective control bandwidth?

If a slower controller reacts strongly to every breath, it may chase a natural cyclic variation.

If it is much slower, breathing appears mainly as a periodic disturbance.

# 18. Breathing as control

The diver can deliberately alter lung volume.

A small deliberate lung-volume change can create a relatively fast but limited buoyancy correction:

\[
\Delta F_B
\approx
\rho g\,u_L.
\]

So breathing can also be treated as a control input:

\[
u_L(t).
\]

A two-input vertical model can be written conceptually as:

\[
m\dot v
=
F_0
+
b_Bu_{\mathrm{BCD}}
+
b_Lu_L
-
mg
-
F_D.
\]

The diver has two different buoyancy-control channels.

# 19. Fast breathing loop, slow BCD loop

A useful conceptual decomposition is:

### Breathing channel

- relatively fast;
- continuously available;
- limited authority;
- naturally coupled to respiration.

### BCD channel

- slower;
- larger persistent buoyancy adjustment;
- changes the baseline gas state.

This suggests **multi-timescale control**.

```text
                  FAST LOOP
depth/velocity ---> breathing ---> small rapid correction
       |
       |
       +----------> BCD ----------> baseline buoyancy
                  SLOW LOOP
```

This is a control-theory model, not a rigid physiological rule.

# 20. Frequency separation

A useful conceptual frequency allocation is:

```text
slow ---------------- medium ---------------- fast
 BCD                  breathing               noise
```

The exact boundaries are not fixed.

The idea is that different mechanisms naturally handle different time scales.

# 21. PID in the frequency domain

Notebook 16 introduced:

\[
C(s)
=
K_P+\frac{K_I}{s}+K_Ds.
\]

At frequency \(\omega\):

### P

Constant gain.

### I

\[
\frac{K_I}{j\omega}
\]

dominates at low frequency.

### D

\[
K_Dj\omega
\]

grows with frequency.

This makes earlier PID observations intuitive:

- integral action is strong against slow persistent error;
- derivative action reacts strongly to fast changes;
- derivative action also amplifies high-frequency noise.

In [ ]:
Kp = 1.0
Ki = 0.3
Kd = 0.2

C_pid = Kp + Ki/(1j*omega_grid) + Kd*(1j*omega_grid)

pid_mag_db = 20*np.log10(np.abs(C_pid))
pid_phase = np.angle(C_pid, deg=True)

plt.semilogx(omega_grid, pid_mag_db)
plt.xlabel("Angular frequency ω [rad/s]")
plt.ylabel("Magnitude [dB]")
plt.title("PID magnitude response")
plt.grid(True)
plt.show()

In [ ]:
plt.semilogx(omega_grid, pid_phase)
plt.xlabel("Angular frequency ω [rad/s]")
plt.ylabel("Phase [degrees]")
plt.title("PID phase response")
plt.grid(True)
plt.show()

# 22. Derivative filtering

An ideal derivative:

\[
K_Ds
\]

has unbounded gain as frequency increases.

A practical filtered derivative can be written:

\[
C_D(s)
=
K_D\frac{Ns}{s+N}.
\]

This limits high-frequency amplification.

In [ ]:
N = 10.0

D_ideal = Kd*1j*omega_grid
D_filtered = Kd*(N*1j*omega_grid)/(1j*omega_grid + N)

plt.semilogx(
    omega_grid,
    20*np.log10(np.abs(D_ideal)),
    label="Ideal derivative"
)

plt.semilogx(
    omega_grid,
    20*np.log10(np.abs(D_filtered)),
    label="Filtered derivative"
)

plt.xlabel("Angular frequency ω [rad/s]")
plt.ylabel("Magnitude [dB]")
plt.title("Derivative filtering")
plt.grid(True)
plt.legend()
plt.show()

# 23. Sensitivity function

For:

\[
L(s)=C(s)G(s),
\]

define:

\[
\boxed{
S(s)=\frac{1}{1+L(s)}
}
\]

and:

\[
\boxed{
T(s)=\frac{L(s)}{1+L(s)}.
}
\]

These describe how feedback shapes disturbances and tracking across frequency.

Very roughly:

- small \(|S(j\omega)|\) means good disturbance rejection there;
- \(T(j\omega)\) describes closed-loop tracking and some measurement pathways.

This is the deeper language behind bandwidth and robustness.

# 24. Why breathing belongs here — but not entirely

Frequency response is the ideal place to introduce breathing because ordinary breathing is naturally periodic.

But breathing deserves its own physical model because it is not simply a sine-wave disturbance.

We still need to model:

- lung-volume state;
- tidal breathing;
- mean lung-volume shift;
- voluntary modulation;
- limits;
- pressure dependence;
- interaction with BCD control.

That is Notebook 18.

# 25. Frequency response connects the course

We can now reinterpret earlier topics:

### Delay
Phase lag.

### Sensor filtering
High-frequency attenuation plus added lag.

### PID
Frequency-selective control action.

### Breathing
Periodic disturbance and fast limited actuator.

### Human controller
Finite bandwidth.

### Stability robustness
Gain margin and phase margin.

# Exercises

### 1. First-order frequency response

For:

\[
G(s)=\frac{1}{2s+1},
\]

compute magnitude and phase at:

\[
\omega=0.1,\ 0.5,\ 1,\ 5.
\]

In [ ]:
# Your code here

### 2. Bandwidth

Compare:

\[
\tau=0.5,\ 1,\ 3.
\]

Plot their Bode magnitudes.

How does time constant affect bandwidth?

In [ ]:
# Your code here

### 3. Delay

For:

\[
T=0.2,\ 0.8,\ 2.0\ \text{s},
\]

plot phase lag versus frequency.

At what frequency does each delay produce \(-90^\circ\)?

In [ ]:
# Your code here

### 4. Breathing frequency

Change the illustrative breathing frequency.

Mark it on the Bode plot.

What amplitude attenuation and phase lag does the example plant have there?

### 5. PID components

Plot separately:

\[
K_P,
\qquad
\frac{K_I}{s},
\qquad
K_Ds.
\]

Which term dominates at low, medium and high frequency?

In [ ]:
# Your code here

# Challenge — breathing disturbance rejection

Choose:

\[
G(s)=\frac{1}{\tau s+1}
\]

and a controller \(C(s)\).

Compute:

\[
S(j\omega)
=
\frac{1}{1+C(j\omega)G(j\omega)}.
\]

Evaluate:

\[
|S(j\omega_b)|
\]

at an illustrative breathing frequency.

Then change the controller parameters.

Discuss why making the controller faster is not always desirable when delay and sensor noise are present.

In [ ]:
# Your code here

# Summary

Frequency response studies:

\[
G(j\omega).
\]

Its magnitude:

\[
|G(j\omega)|
\]

describes amplitude change.

Its phase:

\[
\angle G(j\omega)
\]

describes timing shift.

We introduced:

- sinusoidal steady-state response;
- Bode plots;
- bandwidth;
- delay as phase lag;
- gain and phase margins;
- PID frequency behavior;
- derivative filtering;
- sensitivity.

And we introduced the dual role of breathing:

\[
\boxed{\text{periodic disturbance}}
\]

and:

\[
\boxed{\text{limited fast control input}}.
\]

This suggests a multi-timescale interpretation:

\[
\boxed{
\text{BCD: slower baseline adjustment}
\qquad
\text{breathing: faster limited correction}
}
\]

### Next — Notebook 18

Notebook 18 will develop **Breathing Dynamics** explicitly:

- lung-volume state;
- tidal breathing;
- mean lung-volume control;
- buoyancy contribution;
- pressure effects;
- voluntary control limits;
- interaction between breathing and BCD.